In [2]:
import pandas as pd
import boto3
import time
import matplotlib.pyplot as plt

print("🧠 Initializing AWS Comprehend client...")

comprehend = boto3.client(
    "comprehend",
    region_name="eu-west-1"
)

print("✅ AWS Comprehend client ready")

🧠 Initializing AWS Comprehend client...
✅ AWS Comprehend client ready


In [3]:
print("📄 Loading translated articles...")

df = pd.read_csv("articles_final.csv")

print("✅ Dataset loaded")
print("Columns:", list(df.columns))
print("Total rows:", len(df))

📄 Loading translated articles...
✅ Dataset loaded
Columns: ['Source', 'Title', 'URL', 'Language', 'Unnamed: 4', 'og_text', 'translated_text']
Total rows: 30


In [4]:
del df['Unnamed: 4']

In [6]:
sentiment_cols = [
    "sentiment",
    "sentiment_positive",
    "sentiment_negative",
    "sentiment_neutral",
    "sentiment_mixed"
]

for col in sentiment_cols:
    if col not in df.columns:
        df[col] = pd.NA

In [10]:
import time

print("🔍 Running sentiment analysis with AWS Comprehend...")

for idx, row in df[df["translated_text"].notna()].iterrows():
    try:
        full_text = str(row["translated_text"])
        
        # AWS Comprehend limit is 5,000 bytes. 
        # Using 4900 as a safety margin.
        chunk_limit = 4900 
        
        # Split text into chunks
        chunks = [full_text[i:i + chunk_limit] for i in range(0, len(full_text), chunk_limit)]
        
        chunk_responses = []

        for chunk in chunks:
            if chunk.strip():
                resp = comprehend.detect_sentiment(
                    Text=chunk,
                    LanguageCode="en"
                )
                chunk_responses.append(resp)
        
        if not chunk_responses:
            continue

        # Average the Sentiment Scores
        pos = sum(r["SentimentScore"]["Positive"] for r in chunk_responses) / len(chunk_responses)
        neg = sum(r["SentimentScore"]["Negative"] for r in chunk_responses) / len(chunk_responses)
        neu = sum(r["SentimentScore"]["Neutral"] for r in chunk_responses) / len(chunk_responses)
        mix = sum(r["SentimentScore"]["Mixed"] for r in chunk_responses) / len(chunk_responses)

        # Determine the final Sentiment label based on the highest average score
        scores = {"POSITIVE": pos, "NEGATIVE": neg, "NEUTRAL": neu, "MIXED": mix}
        final_sentiment = max(scores, key=scores.get)

        # Save to DataFrame
        df.at[idx, "sentiment"] = final_sentiment
        df.at[idx, "sentiment_positive"] = pos
        df.at[idx, "sentiment_negative"] = neg
        df.at[idx, "sentiment_neutral"] = neu
        df.at[idx, "sentiment_mixed"] = mix

        print(f"✅ Analyzed row {idx} ({len(chunk_responses)} chunks)")
        
        # Small sleep to respect AWS throttling limits
        time.sleep(0.3)

    except Exception as e:
        print(f"❌ Sentiment failed at row {idx}: {str(e)}")

🔍 Running sentiment analysis with AWS Comprehend...
✅ Analyzed row 0 (2 chunks)
✅ Analyzed row 1 (1 chunks)
✅ Analyzed row 2 (4 chunks)
✅ Analyzed row 3 (2 chunks)
✅ Analyzed row 4 (1 chunks)
✅ Analyzed row 5 (2 chunks)
✅ Analyzed row 6 (2 chunks)
✅ Analyzed row 7 (2 chunks)
✅ Analyzed row 8 (1 chunks)
✅ Analyzed row 9 (1 chunks)
✅ Analyzed row 10 (2 chunks)
✅ Analyzed row 11 (1 chunks)
✅ Analyzed row 12 (1 chunks)
✅ Analyzed row 13 (2 chunks)
✅ Analyzed row 14 (2 chunks)
✅ Analyzed row 15 (2 chunks)
✅ Analyzed row 16 (1 chunks)
✅ Analyzed row 17 (2 chunks)
✅ Analyzed row 18 (2 chunks)
✅ Analyzed row 19 (2 chunks)
✅ Analyzed row 20 (2 chunks)
✅ Analyzed row 21 (1 chunks)
✅ Analyzed row 22 (1 chunks)
✅ Analyzed row 23 (1 chunks)
✅ Analyzed row 24 (1 chunks)
✅ Analyzed row 25 (1 chunks)
✅ Analyzed row 26 (1 chunks)
✅ Analyzed row 27 (3 chunks)
✅ Analyzed row 28 (2 chunks)
✅ Analyzed row 29 (2 chunks)


In [12]:
df.to_csv("articles_with_sentiment.csv", index=False)